# Selective Copy Gate — v2, with the dt-init / lr-scale fix

**Project: `transformer-vs-ssm` (sequence-modeling blog series). Unrelated to any competition.**

## What changed since v1

v1 (this kernel's first version) found Mamba stuck at/below chance (mean acc 2.2%,
transformer 98.3%). Root causes, found by instrumenting gradients rather than retuning:

1. **Delta pinned at init.** A bias-free projection put softplus(~0) = ln(2) into
   every state-decay step regardless of input; combined with A in [-1,-16], state
   half-life was ~1 step. Fixed with a dedicated `dt_proj` using the reference
   (Gu & Dao) inverse-softplus bias init.
2. **That fix was being silently erased** by a model-wide re-init pass that ran
   after construction. Fixed by marking `dt_proj` `_no_reinit`.
3. **Gradients on the selectivity parameters (`dt_proj`, `x_proj`, `A_log`) are
   4-5 orders of magnitude smaller** than the rest of the network -- no single
   global LR can train both. `ssm_lr_scale` in `src/optim.py` boosts exactly
   those parameters. Verified locally: dead loss through step 500, breaking to
   8.8% accuracy (2.7x chance) by step 800 at scale=100 -- still climbing when
   the local check stopped.

This run: (a) a short screen over `ssm_lr_scale` to find a working value at full
step count, (b) the actual gate (selective vs ablated vs transformer, 3 seeds) at
the winning scale.

**Verdict logic** (unchanged from v1)
- selective >> ablated  -> selectivity works; proceed to MQAR sweeps
- selective ~= ablated  -> selection still not functioning; stop and fix further
- transformer also fails -> the *task/training* config is wrong, not the model

## Kaggle settings
- Accelerator: **GPU T4 x1**, Internet **ON**
- Save Version -> Run All

In [ ]:
# Cell 1: Clone (or refresh) the repo -- picks up the a5b9acd fix
import os, subprocess, sys

REPO = "https://github.com/nvaidyan1/transformer-vs-ssm.git"
if not os.path.exists("transformer-vs-ssm"):
    subprocess.run(["git", "clone", REPO], check=True)

os.chdir("/kaggle/working/transformer-vs-ssm")
subprocess.run(["git", "pull", "--ff-only"], check=False)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
# Cell 2: Dependencies
!pip install -q pyyaml pytest 2>&1 | tail -1
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Cell 3: Full test suite must pass, including the new dt-init / lr-scale regression tests.
!python -m pytest tests/ -q 2>&1 | tail -6

In [ ]:
# Cell 4: Confirm the fix locally on the actual model before spending GPU time on it.
import sys, math, torch
sys.path.insert(0, "/kaggle/working/transformer-vs-ssm")
from src.models import build_model
from src.models.mamba import DT_MIN, DT_MAX

m = build_model('mamba', vocab_size=64, n_layers=4, d_model=124, d_state=16,
                d_conv=4, expand=2, dropout=0.0).eval()
x = torch.randint(0, 64, (8, 64))
with torch.no_grad():
    _, deltas = m(x, return_delta=True)
d = torch.cat([t.flatten() for t in deltas])
print(f"delta at init: mean {d.mean():.5f}  (target range [{DT_MIN},{DT_MAX}])")
assert DT_MIN * 0.1 < d.mean().item() < DT_MAX * 10, "delta init looks wrong -- stop here"
print("OK -- delta init confirmed sane before proceeding")

In [ ]:
# Cell 5: Shared settings
TASK_ARGS   = "n_data=4,seq_len=64"
BASE_LR     = 3e-4          # global LR was tested and shown NOT to be the lever
                             # (3e-4 vs 1e-2, both dead) -- do not re-sweep it here
SCALES      = [10, 100, 300]
SCREEN_STEPS = 1500
FINAL_STEPS  = 4000
BATCH        = 32
SEEDS        = [0, 1, 2]
OUT          = "/kaggle/working/gate_v2"

import os
os.makedirs(OUT, exist_ok=True)
print(f"task_args={TASK_ARGS}  base_lr={BASE_LR}  scales={SCALES}  "
      f"screen_steps={SCREEN_STEPS}  final_steps={FINAL_STEPS}  seeds={SEEDS}")

In [ ]:
# Cell 6: Screen ssm_lr_scale -- one seed each, selective only, reduced steps.
# Picks the scale to use for the full multi-seed gate in Cell 8-9.
for scale in SCALES:
    !python src/train_task.py --config configs/recall/mamba_small.yaml \
        --task selective_copy --task_args {TASK_ARGS} --steps {SCREEN_STEPS} \
        --batch_size {BATCH} --lr {BASE_LR} --ssm_lr_scale {scale} --seed 0 \
        --device cuda --out {OUT}/screen_scale{scale}.json 2>&1 | tail -4

In [ ]:
# Cell 7: Pick the winning scale
import glob, json

screen = {}
for f in sorted(glob.glob(f"{OUT}/screen_scale*.json")):
    r = json.load(open(f))
    screen[r["ssm_lr_scale"]] = r["final_val_acc"]

print(f"{'scale':>8} {'val_acc':>9}")
for s, a in sorted(screen.items()):
    print(f"{s:>8} {a:>9.4f}")

BEST_SCALE = max(screen, key=screen.get)
print(f"\nBEST_SCALE = {BEST_SCALE}  (val_acc {screen[BEST_SCALE]:.4f} at "
      f"{SCREEN_STEPS} steps -- final gate runs {FINAL_STEPS})")

In [ ]:
# Cell 8: Mamba -- SELECTIVE, at the winning scale, full steps, 3 seeds
for s in SEEDS:
    !python src/train_task.py --config configs/recall/mamba_small.yaml \
        --task selective_copy --task_args {TASK_ARGS} --steps {FINAL_STEPS} \
        --batch_size {BATCH} --lr {BASE_LR} --ssm_lr_scale {BEST_SCALE} --seed {s} \
        --device cuda --out {OUT}/mamba_selective_seed{s}.json 2>&1 | tail -4

In [ ]:
# Cell 9: Mamba -- ABLATED, same scale and steps (control: no position-wise selectivity)
for s in SEEDS:
    !python src/train_task.py --config configs/recall/mamba_small.yaml \
        --task selective_copy --task_args {TASK_ARGS} --steps {FINAL_STEPS} \
        --batch_size {BATCH} --lr {BASE_LR} --ssm_lr_scale {BEST_SCALE} --seed {s} \
        --device cuda --ablate \
        --out {OUT}/mamba_ablated_seed{s}.json 2>&1 | tail -4

In [ ]:
# Cell 10: Transformer -- positive control on the training setup
for s in SEEDS:
    !python src/train_task.py --config configs/recall/transformer_small.yaml \
        --task selective_copy --task_args {TASK_ARGS} --steps {FINAL_STEPS} \
        --batch_size {BATCH} --seed {s} --device cuda \
        --out {OUT}/transformer_seed{s}.json 2>&1 | tail -4

In [ ]:
# Cell 11: Verdict
import glob, json
import numpy as np

rows = {}
for f in sorted(glob.glob(f"{OUT}/*seed*.json")):
    r = json.load(open(f))
    key = f"{r['arch']}{' (ablated)' if r['ablate'] else ''}"
    rows.setdefault(key, []).append(r["final_val_acc"])

chance = json.load(open(glob.glob(f"{OUT}/*seed*.json")[0]))["chance_acc"]
print(f"BEST_SCALE = {BEST_SCALE}\n")
print(f"{'run':24s} {'mean acc':>9s} {'min':>7s} {'max':>7s}   (chance {chance:.3f})")
print("-" * 58)
for k, v in sorted(rows.items()):
    print(f"{k:24s} {np.mean(v):>9.4f} {np.min(v):>7.4f} {np.max(v):>7.4f}")

sel = np.mean(rows.get("mamba", [0]))
abl = np.mean(rows.get("mamba (ablated)", [0]))
tra = np.mean(rows.get("transformer", [0]))

print()
if tra < 0.5:
    print("INCONCLUSIVE -- the transformer also failed. The task/training config is at "
          "fault, not the Mamba implementation.")
elif sel > 0.9 and sel - abl > 0.2:
    print(f"PASS -- selective {sel:.3f} vs ablated {abl:.3f}. Selectivity is functioning; "
          "MQAR results from this implementation are interpretable.")
elif sel - abl < 0.1:
    print(f"FAIL -- selective {sel:.3f} ~= ablated {abl:.3f}. Selection is still not doing "
          "work at scale={BEST_SCALE}. Consider more steps, a different scale, or that "
          "something else remains broken.")
else:
    print(f"PARTIAL -- selective {sel:.3f}, ablated {abl:.3f}, transformer {tra:.3f}. "
          "Real improvement over v1 (2.2%) but not yet a clean pass -- inspect curves, "
          "consider more FINAL_STEPS.")

In [ ]:
# Cell 12: Save results as notebook output
import shutil
shutil.make_archive("/kaggle/working/selective_copy_gate_v2_results", "zip", OUT)
print("saved -> /kaggle/working/selective_copy_gate_v2_results.zip")
!ls -la /kaggle/working/*.zip